In [ ]:
'''For GCA dataset'''

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.svm import SVC, SVR
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from skrebate import ReliefF  # pip install skrebate — real ReliefF
from itertools import combinations
from scipy.stats import ttest_ind
import csv
df = pd.read_csv('../data/GSE201753/normalized_data_deg.csv')
print(df.shape)


labels_df = pd.read_csv("../data/GSE201753/GSE201753_metadata.csv", index_col=0)
labels = labels_df['condition']

first_col_name = df.columns[0]
first_col = df.iloc[:, 0]

# 1. Isolate the numeric data first
df_numeric = df.iloc[:, 1:]

# 2. Calculate variance ONLY on the numeric columns
gene_var = df_numeric.var(axis=0)
low_var_threshold = gene_var.quantile(0.20)  # Bottom 20% variance threshold

# 3. Filter the numeric columns using the correctly aligned mask
df_filtered = df_numeric.loc[:, gene_var >= low_var_threshold]

# 4. Insert the first column back at the beginning
df_filtered.insert(0, first_col_name, first_col)
df = df_filtered

print(f"Data shape after filtering low-variance genes further: {df.shape}  (samples × genes)")

df.to_csv('normalized_and_filtered_data_deg.csv', index=False)

df.head()

# ============================================================
# Build X (samples x genes) and y, aligned by sample ID
# ============================================================
y_raw = df[first_col_name]                       # first column = 'condition' labels
X = df.drop(columns=first_col_name)              # remaining columns = genes
y = LabelEncoder().fit_transform(y_raw)          # numeric labels (ReliefF-safe)

print(X.shape, np.unique(y_raw, return_counts=True))


(109, 3683)
Data shape after filtering low-variance genes further: (109, 2946)  (samples × genes)


/var/folders/mk/dhch81x172qcjg2g_ztdf3qw0000gn/T/ipykernel_6167/2955783846.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_filtered.insert(0, first_col_name, first_col)


(109, 2945) (array(['ACT', 'HD', 'RNT', 'RT'], dtype=object), array([19, 29, 28, 33]))


In [ ]:
import pandas as pd

def build_rank_table(feature_names, score_dict):
    names = np.asarray(feature_names)
    cols = {'Rank': np.arange(1, len(names) + 1)}
    for method, scores in score_dict.items():
        scores = np.asarray(scores)
        order = np.argsort(scores, kind='stable')[::-1]    # best first
        cols[f'{method} Gene']  = names[order]
        cols[f'{method} Score'] = scores[order]
    return pd.DataFrame(cols)

In [ ]:
def rank_all_features(X, y, rfe_step=0.01, verbose=True):
    """
    Runs every method once on the full dataset and returns a table with a
    Gene/Score column pair per method, covering every gene, best to worst.

    rfe_step : fraction of remaining genes SVM-RFE drops per iteration.
               Smaller = finer ranking (fewer ties) but slower.
    """
    y = np.asarray(y)

    # ── Scale + drop constant genes ──────────────────────────
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    feature_names = np.asarray(X.columns)

    if verbose:
        print(f"Genes before near-zero variance filter: {X_scaled.shape[1]}")
    valid = X_scaled.std(axis=0) > 0.01
    X_scaled = X_scaled[:, valid]
    feature_names = feature_names[valid]
    if verbose:
        print(f"Genes after near-zero variance filter:  {X_scaled.shape[1]}")

    n_genes = X_scaled.shape[1]

    # ── 1. Lasso ─────────────────────────────────────────────
    lasso = Lasso(alpha=0.01, max_iter=5000)
    lasso.fit(X_scaled, y)
    lasso_scores = np.abs(lasso.coef_)

    # ── 2. SVM-RFE (n_features_to_select=1 -> full elimination order) ──
    rfe = RFE(SVC(kernel='linear', class_weight='balanced'),
              n_features_to_select=1, step=rfe_step)
    rfe.fit(X_scaled, y)
    rfe_scores = -rfe.ranking_                   # rank 1 = best, so negate

    # ── 3. Random Forest ─────────────────────────────────────
    rf = RandomForestClassifier(
        n_estimators=500, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_scaled, y)
    rf_scores = rf.feature_importances_

    # ── 4. SVM-RBF (correlation with prediction confidence) ──
    svm_rbf = SVC(kernel='rbf', class_weight='balanced', probability=True)
    svm_rbf.fit(X_scaled, y)
    max_conf = svm_rbf.predict_proba(X_scaled).max(axis=1)

    rbf_scores = np.nan_to_num(np.array([
        abs(np.corrcoef(X_scaled[:, i], max_conf)[0, 1])
        for i in range(n_genes)
    ]))

    # ── 5. Double-RBF (two-stage) ────────────────────────────
    stage1_idx = np.argsort(rbf_scores)[::-1][:n_genes // 2]
    X_stage1 = X_scaled[:, stage1_idx]

    svm_rbf2 = SVC(kernel='rbf', class_weight='balanced', probability=True)
    svm_rbf2.fit(X_stage1, y)
    conf2 = svm_rbf2.predict_proba(X_stage1).max(axis=1)

    rbf2_stage = np.nan_to_num(np.array([
        abs(np.corrcoef(X_stage1[:, i], conf2)[0, 1])
        for i in range(X_stage1.shape[1])
    ]))
    rbf2_scores = np.zeros(n_genes)              # genes cut in stage 1 stay 0
    rbf2_scores[stage1_idx] = rbf2_stage

    # ── 6. ReliefF ───────────────────────────────────────────
    relief = ReliefF(n_features_to_select=n_genes, n_neighbors=10)
    relief.fit(X_scaled, y)
    relief_scores = relief.feature_importances_

    # ── 7. SigFeature (SVM weights x t-stat) ─────────────────
    ovr = OneVsRestClassifier(SVC(kernel='linear', class_weight='balanced'))
    ovr.fit(X_scaled, y)
    svm_weights = np.abs(np.vstack([e.coef_ for e in ovr.estimators_])).mean(axis=0)

    classes_unique = np.unique(y)
    t_stats = np.zeros(n_genes)
    for i in range(n_genes):
        pair_stats = []
        for c1, c2 in combinations(classes_unique, 2):
            g1 = X_scaled[y == c1, i]
            g2 = X_scaled[y == c2, i]
            if len(g1) > 1 and len(g2) > 1:
                t, _ = ttest_ind(g1, g2)
                pair_stats.append(abs(t))
        t_stats[i] = np.mean(pair_stats) if pair_stats else 0
    sig_scores = svm_weights * t_stats

    # ── 8. Ensemble (Borda count over the 7 methods) ─────────
    borda_scores = np.zeros(n_genes)
    for s in (lasso_scores, rf_scores, rfe_scores, rbf_scores,
              rbf2_scores, sig_scores, relief_scores):
        borda_scores += np.argsort(np.argsort(s))     # rank 0 = worst

    # ── 9. Ranked table (column order = your spreadsheet) ────
    score_dict = {
        'LASSO':         lasso_scores,
        'Double-RBF':    rbf2_scores,
        'SVM_RBF':       rbf_scores,
        'SVM-RFE':       rfe_scores,
        'Ensemble':      borda_scores,
        'Random Forest': rf_scores,
        'ReliefF':       relief_scores,
        'SigFeature':    sig_scores,
    }
    return build_rank_table(feature_names, score_dict)

In [ ]:
table = rank_all_features(X, y)
table.to_excel("GCA_Feature_Compilation.xlsx", index=False)
print(table.shape)
table.head()